In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn


def train_and_give_test_accuracy(train_data: pd.DataFrame, test_data: pd.DataFrame) -> float:
    

    seed = 42
    np.random.seed(seed)
    torch.manual_seed(seed)

    train_data = train_data.drop_duplicates().copy()
    test_data = test_data.copy()

    X_train_df = train_data.drop(columns=['target'])
    y_train_raw = train_data['target']

    X_test_df = test_data.drop(columns=['target'])
    y_test_raw = test_data['target']

    # Encodage de la cible en 0/1.
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train_raw)
    y_test = label_encoder.transform(y_test_raw)

    # Pretraitement coherent train/test.
    numeric_cols = X_train_df.select_dtypes(include=np.number).columns
    categorical_cols = X_train_df.columns.difference(numeric_cols)

    X_train_num = X_train_df[numeric_cols].copy()
    X_test_num = X_test_df[numeric_cols].copy()

    medians = X_train_num.median()
    X_train_num = X_train_num.fillna(medians)
    X_test_num = X_test_num.fillna(medians)

    if len(categorical_cols) > 0:
        X_train_cat = X_train_df[categorical_cols].copy()
        X_test_cat = X_test_df[categorical_cols].copy()

        modes = X_train_cat.mode(dropna=True)
        for col in categorical_cols:
            fill_value = modes[col].iloc[0] if not modes[col].empty else 'missing'
            X_train_cat[col] = X_train_cat[col].fillna(fill_value).astype(str)
            X_test_cat[col] = X_test_cat[col].fillna(fill_value).astype(str)

        X_train_cat = pd.get_dummies(X_train_cat, drop_first=False)
        X_test_cat = pd.get_dummies(X_test_cat, drop_first=False)
        X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

        X_train_processed = pd.concat([X_train_num, X_train_cat], axis=1)
        X_test_processed = pd.concat([X_test_num, X_test_cat], axis=1)
    else:
        X_train_processed = X_train_num
        X_test_processed = X_test_num

    variance_filter = VarianceThreshold()
    scaler = StandardScaler()

    X_train_array = variance_filter.fit_transform(X_train_processed)
    X_test_array = variance_filter.transform(X_test_processed)

    X_train_array = scaler.fit_transform(X_train_array)
    X_test_array = scaler.transform(X_test_array)

    # Approximation du noyau RBF par Random Fourier Features.
    gamma = 0.05
    n_components = 512
    lam = 0.001
    lr = 0.01
    epochs = 300

    rng = np.random.default_rng(seed)
    W = rng.normal(
        loc=0.0,
        scale=np.sqrt(2 * gamma),
        size=(X_train_array.shape[1], n_components)
    )
    b = rng.uniform(0, 2 * np.pi, size=n_components)

    def rbf_random_features(X):
        return np.sqrt(2.0 / n_components) * np.cos(X @ W + b)

    Z_train = rbf_random_features(X_train_array)
    Z_test = rbf_random_features(X_test_array)

    # SVM lineaire entraine en PyTorch avec perte hinge.
    model = nn.Linear(n_components, 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_tensor = torch.tensor(Z_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32)

    for _ in range(epochs):
        scores = model(X_tensor).squeeze(1)
        y_svm = 2 * y_tensor - 1
        hinge_loss = torch.clamp(1 - y_svm * scores, min=0).mean()
        regularization = lam * torch.sum(model.weight ** 2)
        loss = hinge_loss + regularization

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        test_scores = model(torch.tensor(Z_test, dtype=torch.float32)).squeeze(1)
        predictions = (test_scores >= 0).numpy().astype(int)

    accuracy = accuracy_score(y_test, predictions)

    return float(accuracy)
    ##  ce code utilise  UN DATASET TRAIN & TEST ET hop CA MARCHE :)